# Model: Evaporator Fouling (Binary Classification)

## Feature choice, informed by notebook 04's EDA

Per notebook 04: RTU_REFG_SUCT_PRES, RTU_REFG_SUCT_TEMP, and RTU_SA_TEMP were all
strong, monotonic signals, filtering/unfiltered alike. Capacity was ALSO strong here
(unlike condenser fouling) - 30% vs 40% Cohen's d = 1.376, "large", confirming the
physical reasoning that the evaporator does the actual cooling work capacity measures.

## Real, testable prediction carried from the emerging pattern

Undercharge (weak signal, comparable to weather noise) generalized poorly forward-
in-time. Overcharge and condenser fouling (both strong signals) generalized well.
Evaporator fouling has the strongest capacity signal found in the Simulated dataset
so far - if the emerging pattern holds, this should generalize at least as well as
condenser fouling, not show undercharge's collapse. Checking directly, not assumed.

In [1]:
import sys
from pathlib import Path

ml_root = Path.cwd().parent
if str(ml_root) not in sys.path:
    sys.path.insert(0, str(ml_root))

from sklearn.ensemble import RandomForestClassifier  # noqa: E402
from sklearn.metrics import classification_report  # noqa: E402
from sklearn.model_selection import TimeSeriesSplit, train_test_split  # noqa: E402
from src.features.build_features import build_feature_table  # noqa: E402

table = build_feature_table(
    baseline_path="../data/raw/RTU_sim_baseline.csv",
    fault_paths={
        "evapfouling10": "../data/raw/RTU_sim_evapfouling10.csv",
        "evapfouling20": "../data/raw/RTU_sim_evapfouling20.csv",
        "evapfouling30": "../data/raw/RTU_sim_evapfouling30.csv",
        "evapfouling40": "../data/raw/RTU_sim_evapfouling40.csv",
        "evapfouling50": "../data/raw/RTU_sim_evapfouling50.csv",
    },
    pressure_temp_cols=("RTU_REFG_SUCT_PRES", "RTU_REFG_SUCT_TEMP", "RTU_SA_TEMP"),
)

print(f"Feature table shape: {table.shape}")
print(f"\nLabel distribution:\n{table['label'].value_counts()}")
table.head()

Feature table shape: (378573, 7)

Label distribution:
label
1    315383
0     63190
Name: count, dtype: int64


,Datetime,label,source_file,RTU_REFG_SUCT_PRES_residual,RTU_REFG_SUCT_TEMP_residual,RTU_SA_TEMP_residual,RTU_TOT_CAPA_ewma30_segmented_residual
0,2018-07-20 01:00:00,0,baseline,-9.972298e+03,0.704927,0.270441,662.149504
1,2018-07-20 01:00:00,1,evapfouling20,-5.984073e+05,-1.787871,-2.352164,-18.051496
2,2018-07-20 01:00:00,1,evapfouling30,-9.691113e+05,-3.400008,-4.042681,-431.146496
3,2018-07-20 01:00:00,1,evapfouling40,-1.437131e+06,-5.484553,-6.223139,-969.773496
4,2018-07-20 01:00:00,1,evapfouling50,-2.043114e+06,-8.270300,-9.128252,-1695.246496


## Evaluating evaporator fouling: both random-split and TimeSeriesSplit

In [2]:

feature_cols = [
    "RTU_REFG_SUCT_PRES_residual",
    "RTU_REFG_SUCT_TEMP_residual",
    "RTU_SA_TEMP_residual",
    "RTU_TOT_CAPA_ewma30_segmented_residual",
]

X_all = table[feature_cols].values
y_all = table["label"].values

X_tr_r, X_te_r, y_tr_r, y_te_r = train_test_split(
    X_all, y_all, test_size=0.2, random_state=42, stratify=y_all
)
rf_random = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
rf_random.fit(X_tr_r, y_tr_r)
y_pred_r = rf_random.predict(X_te_r)

print("=== Random split ===")
print(classification_report(y_te_r, y_pred_r, target_names=["baseline", "evapfouling"]))

tscv = TimeSeriesSplit(n_splits=5)
print("=== TimeSeriesSplit (5 folds) ===")
for fold_num, (train_idx, test_idx) in enumerate(tscv.split(X_all), start=1):
    X_tr, X_te = X_all[train_idx], X_all[test_idx]
    y_tr, y_te = y_all[train_idx], y_all[test_idx]

    fold_model = RandomForestClassifier(n_estimators=100, max_depth=5, random_state=42)
    fold_model.fit(X_tr, y_tr)
    y_pred_fold = fold_model.predict(X_te)

    report = classification_report(y_te, y_pred_fold, target_names=["baseline", "evapfouling"], output_dict=True)
    print(f"Fold {fold_num}: baseline recall={report['baseline']['recall']:.2f}, "
          f"baseline precision={report['baseline']['precision']:.2f}, "
          f"evapfouling recall={report['evapfouling']['recall']:.2f}")

=== Random split ===
              precision    recall  f1-score   support

    baseline       0.90      0.71      0.79     12638
 evapfouling       0.94      0.98      0.96     63077

    accuracy                           0.94     75715
   macro avg       0.92      0.85      0.88     75715
weighted avg       0.94      0.94      0.94     75715

=== TimeSeriesSplit (5 folds) ===
Fold 1: baseline recall=0.76, baseline precision=0.97, evapfouling recall=1.00
Fold 2: baseline recall=0.60, baseline precision=0.96, evapfouling recall=1.00
Fold 3: baseline recall=0.53, baseline precision=0.93, evapfouling recall=0.99
Fold 4: baseline recall=0.45, baseline precision=0.95, evapfouling recall=1.00
Fold 5: baseline recall=0.41, baseline precision=0.96, evapfouling recall=1.00


## Evaporator fouling: a third, distinct pattern — partial degradation, not full
## collapse and not fully stable — complicating the emerging hypothesis

| | Random split | TS Fold 1 | Fold 2 | Fold 3 | Fold 4 | Fold 5 |
|---|---|---|---|---|---|---|
| Baseline recall | 0.71 | 0.76 | 0.60 | 0.53 | 0.45 | 0.41 |
| Baseline precision | 0.90 | 0.97 | 0.96 | 0.93 | 0.95 | 0.96 |

**This does not cleanly fit either prior pattern.** A real, gradual degradation trend
exists (0.76 → 0.41, roughly halving) — unlike condenser fouling's flat 0.99-1.00 —
but it plateaus rather than collapsing to zero like undercharge did, and precision
stays consistently high throughout (never drops below 0.93), meaning the model
isn't becoming confidently wrong, just progressively less sensitive to baseline
over time.

**This complicates the "strong signal → good generalization" hypothesis** from the
condenser fouling notebook: evaporator fouling has arguably the single strongest
capacity signal in the whole Simulated dataset (Cohen's d=1.376 at the toughest
adjacent-severity comparison), yet it still shows real, gradual forward-in-time
degradation. Signal strength alone does not fully predict generalization behavior —
there must be some other factor differentiating undercharge/evaporator fouling
(both show real degradation, to different degrees) from overcharge/condenser fouling
(both stable). Not yet identified. Flagged as a genuinely open question, not
resolved by this notebook — worth a dedicated look once all 6 faults are modeled and
there's a full picture to compare against, rather than guessing now on partial data.

**Practical status**: this is still a usable model (recall 0.41-1.00 depending on
fold, precision consistently 0.93-0.97) but with a real, honestly-reported caveat —
worse than condenser fouling/overcharge, better than undercharge's unresolved
collapse. A three-tier picture is emerging: stable (overcharge, condenser fouling),
gradually degrading (evaporator fouling), and collapsing (undercharge) — not a clean
binary.

## Summary: evaporator fouling binary classifier

A third, distinct generalization pattern - gradual baseline-recall degradation
(0.76→0.41 across folds) that plateaus rather than collapses, with precision staying
high (0.93-0.97) throughout. Complicates the "strong signal generalizes well"
hypothesis from condenser fouling/overcharge - evaporator fouling has the dataset's
strongest capacity signal yet still degrades meaningfully. Genuinely usable model
with an honestly-reported caveat, positioned between condenser fouling's stability
and undercharge's collapse. Real open question about what actually predicts
generalization behavior - not yet resolved, worth revisiting once all 6 faults are
modeled.